# llm-kernel-lib — Development Notebook

Run cells top-to-bottom. Works on Colab T4 (free tier) for development;
run final benchmarks on an A30 for the numbers that go on your resume.

**Sections**
1. Environment setup
2. Flash Attention — correctness + benchmark
3. Fused RMSNorm + Linear — correctness + speedup
4. int8 GEMM — correctness + throughput
5. End-to-end: swap all three into a tiny transformer block


## 1. Environment setup

In [ ]:
# Pin versions — mismatches between Triton / PyTorch / CUDA are the #1 time sink
!pip install -q torch==2.2.0 triton==2.2.0 ninja packaging tabulate

# Clone repo (replace with your fork URL after pushing)
!git clone -q https://github.com/your-handle/llm-kernel-lib
%cd llm-kernel-lib

# Build the CUDA extension (fused RMSNorm+Linear)
# T4 = SM75, A30 = SM86, V100 = SM70
import subprocess, torch
sm = torch.cuda.get_device_capability()
arch = f"{sm[0]}{sm[1]}"
print(f"Building for SM{arch} ({torch.cuda.get_device_name(0)})")
!TORCH_CUDA_ARCH_LIST="{arch[0]}.{arch[1]}" pip install -e . --no-build-isolation -q
print('Build complete')

In [ ]:
import torch
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.version.cuda}')
print(f'GPU      : {torch.cuda.get_device_name(0)}')
print(f'VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

import sys
sys.path.insert(0, '.')
import llm_kernels
print(f'llm_kernels loaded OK')

## 2. Flash Attention

In [ ]:
# ── Correctness ───────────────────────────────────────────────────────────
import torch, llm_kernels
torch.manual_seed(0)

B, H, M, D = 2, 8, 512, 64
q = torch.randn(B, H, M, D, device='cuda', dtype=torch.float16)
k = torch.randn_like(q)
v = torch.randn_like(q)

ref = torch.nn.functional.scaled_dot_product_attention(q, k, v, is_causal=True)
out = llm_kernels.flash_attn_forward(q, k, v, causal=True)

max_err = (ref - out).abs().max().item()
print(f'Max absolute error vs torch SDPA: {max_err:.4f}  (atol=0.01 expected)')
assert max_err < 0.02, 'Correctness check failed!'
print('PASSED')

In [ ]:
# ── Benchmark ─────────────────────────────────────────────────────────────
import time

def bench(fn, warmup=10, reps=100):
    for _ in range(warmup): fn()
    torch.cuda.synchronize()
    t = time.perf_counter()
    for _ in range(reps): fn()
    torch.cuda.synchronize()
    return (time.perf_counter() - t) * 1e3 / reps  # ms

print(f'{"seqlen":>8}  {"Triton FA":>12}  {"torch SDPA":>12}  {"TFLOP/s":>10}  {"ratio":>8}')
print('-' * 60)
B, H, D = 2, 16, 64
for M in [512, 1024, 2048, 4096]:
    q = torch.randn(B,H,M,D, device='cuda', dtype=torch.float16)
    k, v = torch.randn_like(q), torch.randn_like(q)
    flops = 4 * B * H * M * M * D

    ms_t = bench(lambda: llm_kernels.flash_attn_forward(q, k, v, causal=True))
    ms_r = bench(lambda: torch.nn.functional.scaled_dot_product_attention(q, k, v, is_causal=True))

    tflops = flops / ms_t / 1e9
    ratio  = ms_r / ms_t
    print(f'{M:>8}  {ms_t:>10.3f}ms  {ms_r:>10.3f}ms  {tflops:>10.1f}  {ratio:>7.2f}x')

## 3. Fused RMSNorm + Linear

In [ ]:
# ── Correctness ───────────────────────────────────────────────────────────
torch.manual_seed(1)
B, D_in, D_out = 512, 4096, 4096
x      = torch.randn(B, D_in,        device='cuda', dtype=torch.float16)
w_norm = torch.ones(D_in,            device='cuda', dtype=torch.float16)
w_lin  = torch.randn(D_out, D_in,    device='cuda', dtype=torch.float16)

# Reference: separate PyTorch ops
rms = torch.nn.RMSNorm(D_in, device='cuda', dtype=torch.float16)
lin = torch.nn.Linear(D_in, D_out, bias=False, device='cuda', dtype=torch.float16)
with torch.no_grad():
    rms.weight.copy_(w_norm)
    lin.weight.copy_(w_lin)
ref = lin(rms(x))

out = llm_kernels.fused_rmsnorm_linear(x, w_norm, w_lin)
max_err = (ref.float() - out.float()).abs().max().item()
print(f'Max absolute error vs unfused PyTorch: {max_err:.4f}  (atol=0.1 expected)')
assert max_err < 0.15, 'Correctness check failed!'
print('PASSED')

In [ ]:
# ── Speedup benchmark ─────────────────────────────────────────────────────
print(f'{"config":>30}  {"unfused":>10}  {"fused":>10}  {"speedup":>8}')
print('-' * 65)
for B, D in [(128, 8192), (512, 4096), (1024, 2048)]:
    x = torch.randn(B, D, device='cuda', dtype=torch.float16)
    w_n = torch.ones(D,    device='cuda', dtype=torch.float16)
    w_l = torch.randn(D, D, device='cuda', dtype=torch.float16)
    rms_l = torch.nn.RMSNorm(D, device='cuda', dtype=torch.float16)
    lin_l = torch.nn.Linear(D, D, bias=False, device='cuda', dtype=torch.float16)

    ms_u = bench(lambda: lin_l(rms_l(x)))
    ms_f = bench(lambda: llm_kernels.fused_rmsnorm_linear(x, w_n, w_l))
    print(f'B={B:4d} D={D:4d}  {ms_u:>10.3f}ms  {ms_f:>10.3f}ms  {ms_u/ms_f:>7.2f}x')

## 4. int8 GEMM + dequant

In [ ]:
# ── Correctness ───────────────────────────────────────────────────────────
torch.manual_seed(2)
M = N = K = 512
A = torch.randn(M, K, device='cuda', dtype=torch.float16)
B_mat = torch.randn(K, N, device='cuda', dtype=torch.float16)

A_q, sa = llm_kernels.quantize_symmetric(A)
B_q, sb = llm_kernels.quantize_symmetric(B_mat.T)
B_q = B_q.T.contiguous()

out = llm_kernels.int8_gemm_dequant_fwd(A_q, B_q, sa, sb)
ref = (A @ B_mat).to(torch.float16)

max_err = (ref.float() - out.float()).abs().max().item()
rel_err = ((ref.float() - out.float()).abs() / (ref.float().abs() + 1e-6)).mean().item()
print(f'Max absolute error: {max_err:.3f}')
print(f'Mean relative error: {rel_err*100:.2f}%  (< 1% expected for int8 absmax quant)')
print('PASSED' if max_err < 2.0 else 'FAILED')

In [ ]:
# ── TOPS benchmark ────────────────────────────────────────────────────────
print(f'{"size":>12}  {"int8 TOPS":>12}  {"fp16 TOPS":>12}  {"ratio":>8}')
print('-' * 52)
for size in [1024, 2048, 4096]:
    M = N = K = size
    A_fp = torch.randn(M, K, device='cuda', dtype=torch.float16)
    B_fp = torch.randn(K, N, device='cuda', dtype=torch.float16)
    A_q2, sa2 = llm_kernels.quantize_symmetric(A_fp)
    B_q2, sb2 = llm_kernels.quantize_symmetric(B_fp.T)
    B_q2 = B_q2.T.contiguous()
    ops = 2 * M * N * K

    ms_i = bench(lambda: llm_kernels.int8_gemm_dequant_fwd(A_q2, B_q2, sa2, sb2))
    ms_f = bench(lambda: A_fp @ B_fp)

    tops_i = ops / ms_i / 1e9
    tops_f = ops / ms_f / 1e9
    print(f'M=N=K={size}  {tops_i:>10.1f}T  {tops_f:>10.1f}T  {tops_i/tops_f:>7.2f}x')

## 5. End-to-end: all three kernels in a transformer block

This shows that the nn.Module wrappers compose correctly in real model code.

In [ ]:
import torch.nn as nn

class CustomTransformerBlock(nn.Module):
    """
    One transformer decoder block using all three custom kernels:
      - FlashAttention  (causal self-attention)
      - FusedRMSNormLinear  (pre-norm + up-projection in FFN)
      - Int8Linear  (down-projection in FFN, weight-only int8)
    """
    def __init__(self, d_model: int, n_heads: int, d_ff: int):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model  = d_model
        self.n_heads  = n_heads
        self.d_head   = d_model // n_heads

        # Q, K, V projections (standard linear — kept simple here)
        self.qkv_proj = nn.Linear(d_model, 3 * d_model, bias=False, dtype=torch.float16)
        self.out_proj = nn.Linear(d_model, d_model,     bias=False, dtype=torch.float16)

        # Attention
        self.attn = llm_kernels.FlashAttention(causal=True)

        # FFN: fused norm+up then int8 down
        self.ffn_up   = llm_kernels.FusedRMSNormLinear(d_model, d_ff)
        self.ffn_down = llm_kernels.Int8Linear(d_ff, d_model)
        self.ffn_down.quantize_weights(
            torch.randn(d_model, d_ff, dtype=torch.float16)
        )

        self.norm_attn = nn.RMSNorm(d_model, dtype=torch.float16)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, M, D]
        B, M, D = x.shape

        # ── Self-attention ────────────────────────────────────────────────
        residual = x
        x_norm = self.norm_attn(x)   # pre-norm
        qkv = self.qkv_proj(x_norm).reshape(B, M, 3, self.n_heads, self.d_head)
        q, k, v = qkv.unbind(dim=2)  # each [B, M, H, D_head]
        q = q.transpose(1, 2)        # [B, H, M, D_head]
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        attn_out = self.attn(q, k, v)                         # [B, H, M, D_head]
        attn_out = attn_out.transpose(1, 2).reshape(B, M, D)  # [B, M, D]
        x = residual + self.out_proj(attn_out)

        # ── FFN ───────────────────────────────────────────────────────────
        residual = x
        x_flat = x.reshape(B * M, D)
        up   = self.ffn_up(x_flat)    # fused RMSNorm + linear: [B*M, D_ff]
        up   = torch.nn.functional.gelu(up)
        down = self.ffn_down(up)       # int8 linear: [B*M, D]
        x = residual + down.reshape(B, M, D)

        return x


# ── Test the full block ────────────────────────────────────────────────────
torch.manual_seed(42)
D_MODEL, N_HEADS, D_FF = 512, 8, 2048
B, M = 2, 128

block = CustomTransformerBlock(D_MODEL, N_HEADS, D_FF).cuda()
x = torch.randn(B, M, D_MODEL, device='cuda', dtype=torch.float16)

with torch.no_grad():
    out = block(x)

print(f'Input : {x.shape}  {x.dtype}')
print(f'Output: {out.shape}  {out.dtype}')
assert out.shape == x.shape, 'Shape mismatch!'
assert not out.isnan().any(), 'NaNs in output!'
print('End-to-end test PASSED')